In [2]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

In [3]:
load_dotenv()

True

In [4]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")

ENV VAR LOADED


In [6]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

loader = TextLoader(DATA_FILE_PATH,encoding="utf-8")
documents = loader.load()
print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data/hr_policy.txt
Total characters in document: 2597

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


In [7]:
len(documents)

1

In [8]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [10]:
print(documents[0].metadata)

{'source': 'data/hr_policy.txt'}


In [11]:
print(f"Total characters in document: {len(documents[0].page_content)}")


Total characters in document: 2597


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to

In [ ]:
len(chunks)

9

In [14]:

print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data/hr_policy.txt'}


In [15]:
print(chunks[7].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [16]:
from langchain_community.embeddings import JinaEmbeddings
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)


EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


In [18]:
from langchain_community.vectorstores import FAISS 
vector_store = FAISS.from_documents(chunks , embeddings_model)
print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [20]:
test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [21]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [23]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

llm.model_name

'openai/gpt-oss-120b'

In [24]:
tr = llm.invoke("Hey what is the leave policy")

In [25]:

tr.content

"Sure thing! Could you let me know which company's (or organization's) leave policy you’re interested in? That way I can give you the most relevant details."

In [26]:
from langchain.agents import create_agent

In [27]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)
print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [28]:
def ask_hr_assistant(question:str)->str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)
    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [29]:

response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [30]:
response

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='7801b9cf-41ee-40cb-abc0-9e3e5eba6291'),
  AIMessage(content='I’m the friendly HR assistant here at **Acme\u202fCrop**. How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is a friendly HR assistant working for Acme Crop. Should answer that. No need to search policy. It\'s not about policy. So answer directly.'}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 208, 'total_tokens': 285, 'completion_time': 0.16278777, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.010332011, 'prompt_tokens_details': None, 'queue_time': 0.397870996, 'total_time': 0.173119781}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run-

In [31]:
response["messages"][-2].content

'tell me which org you work for'

In [32]:
print(response["messages"][-1].content)

I’m the friendly HR assistant here at **Acme Crop**. How can I help you today?


In [33]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me about leave policies how to apply for leave"
            }
        ]
    }
)

In [35]:
print(response["messages"][-1].content)

**Leave Policies (Acme Crop)**  

| Type of Leave | Entitlement | Key Conditions |
|---------------|------------|----------------|
| **Annual (Paid) Leave** | 20 days per calendar year | • Must be requested through the HR portal **at least 5 working days before the intended start date**.<br>• Up to **5 unused days** can be carried forward to the next year; any additional unused days are forfeited. |
| **Sick Leave** | 10 paid days per year | • Separate from annual leave.<br>• A **medical certificate** is required if the sick leave exceeds **2 consecutive days**. |
| **Public Holiday** | 12 days per year (as per the official holiday calendar) | • If you work on a public holiday, you are eligible for compensatory leave. |

**How to Apply for Leave**

1. **Log into the HR portal** – Use your employee credentials to access the portal.  
2. **Navigate to the “Leave Request” section** – Choose the appropriate leave type (Annual, Sick, etc.).  
3. **Enter the details** –  
   * Start and end 